In [1]:
import pandas as pd
import numpy as np

In [5]:
np.random.seed(42)

# === PARAMETRY — zmień tutaj ===
N_NORMAL = 2000      # liczba normalnych transakcji
N_FRAUD  = 100       # liczba fraudów
# ===============================

# Normalne transakcje
normal = pd.DataFrame({
    'amount': np.random.lognormal(5, 1, N_NORMAL).clip(5, 5000),
    'is_electronics': np.random.binomial(1, 0.3, N_NORMAL),
    'tx_per_minute': np.random.poisson(3, N_NORMAL),
    'fraud': 0
})


# Fraudy
fraud = pd.DataFrame({
    'amount': np.random.uniform(2000, 9000, N_FRAUD),
    'is_electronics': np.random.binomial(1, 0.7, N_FRAUD),
    'tx_per_minute': np.random.poisson(8, N_FRAUD),
    'fraud': 1
})

df = pd.concat([normal, fraud], ignore_index=True).sample(frac=1, random_state=42)
print(f"Dataset: {len(df)} wierszy, fraud rate: {df['fraud'].mean():.1%}")

Dataset: 2100 wierszy, fraud rate: 4.8%


In [6]:
df.head()

,amount,is_electronics,tx_per_minute,fraud
1034,91.543433,0,3,0
1176,27.304272,0,4,0
67,404.856587,0,2,0
1330,685.913024,0,4,0
650,942.896099,0,2,0


In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pickle

In [8]:
features = ['amount', 'is_electronics', 'tx_per_minute']
X = df[features]
y = df['fraud']

In [9]:
X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size=0.2, random_state = 42, stratify = y)

In [11]:
clf = RandomForestClassifier()

In [12]:
clf.fit(X_train, Y_train)

RandomForestClassifier()

In [18]:
y_pred = clf.predict(X_test)

In [19]:
print(classification_report(Y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       400
           1       1.00      1.00      1.00        20

    accuracy                           1.00       420
   macro avg       1.00      1.00      1.00       420
weighted avg       1.00      1.00      1.00       420



In [20]:
import pickle


In [21]:
with open('fraud_model.pkl', 'wb') as f:
    pickle.dump(clf, f)

In [22]:
model = pickle.load(open('fraud_model.pkl', 'rb'))

In [23]:
model

RandomForestClassifier()

In [24]:
model.predict(X_train)

array([0, 0, 0, ..., 0, 0, 1])

In [25]:
%%file fraud_api.py
from fastapi import FastAPI
from pydantic import BaseModel
import pickle, numpy as np

app = FastAPI(title="Fraud Detection API")
model = pickle.load(open('fraud_model.pkl', 'rb'))

class Transaction(BaseModel):
    amount: float
    is_electronics: int
    tx_per_minute: int

@app.get("/health")
def health():
    return {"status": "ok"}
 
 
@app.post("/score")
def score(tx: Transaction):
    X = np.array([[tx.amount, tx.is_electronics, tx.tx_per_minute]])
    proba = model.predict_proba(X)[0,1]
    return {"is_fraud": bool(proba >= 0.5), "fraud_probability": round(float(proba),4)}

Writing fraud_api.py


In [26]:
import requests

# Test normalna
r = requests.post("http://localhost:8001/score",
    json={"amount": 150, "is_electronics": 0, "tx_per_minute": 3})
print("Normalna:", r.json())

Normalna: {'is_fraud': False, 'fraud_probability': 0.0}


In [1]:
%%file ml_consumer.py
from kafka import KafkaConsumer, KafkaProducer
from datetime import datetime
import json, requests

consumer = KafkaConsumer('transactions', bootstrap_servers='broker:9092',
    auto_offset_reset='earliest', group_id='ml-scoring',
    value_deserializer=lambda x: json.loads(x.decode('utf-8')))

alert_producer = KafkaProducer(bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8'))

API_URL = "http://localhost:8001/score"

from kafka import KafkaConsumer, KafkaProducer
from datetime import datetime
import json
import requests

# === Kafka consumer ===
consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='ml-scoring',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

# === Kafka producer (alerts) ===
alert_producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

API_URL = "http://localhost:8001/score"

print("🚀 ML Consumer started...")

# === GŁÓWNA PĘTLA ===
for message in consumer:
    tx = message.value
    print(f"\nOtrzymano transakcję: {tx}")

    try:
        # 1. Wyciąganie cech
        amount = tx.get("amount", 0)
        is_electronics = tx.get("is_electronics", 0)

        # timestamp → godzina
        timestamp = tx.get("timestamp", None)
        if timestamp:
            hour = datetime.fromisoformat(timestamp).hour
        else:
            hour = 12  # fallback

        # uproszczenie (zgodnie z zadaniem)
        tx_per_minute = 5

        features = {
            "amount": amount,
            "is_electronics": is_electronics,
            "tx_per_minute": tx_per_minute
        }

        # 2. Wywołanie API
        response = requests.post(API_URL, json=features)

        if response.status_code != 200:
            print("Błąd API:", response.text)
            continue

        result = response.json()
        print("Wynik modelu:", result)

        # 3. Jeśli fraud → wysyłamy alert
        if result.get("is_fraud"):
            alert = {
                "timestamp": datetime.now().isoformat(),
                "transaction": tx,
                "fraud_probability": result.get("fraud_probability")
            }

            alert_producer.send('alerts', alert)
            alert_producer.flush()

            print("ALERT! Fraud wykryty:", alert)

    except Exception as e:
        print("❌ Błąd przetwarzania:", e)

Writing ml_consumer.py


In [2]:
import requests
print(requests.get("http://localhost:8001/health").json())

{'status': 'ok'}


In [4]:
clf(y_test, y_pred)

NameError: name 'clf' is not defined